# Loading tools and data

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance


In [ ]:
train_data_to_5 = pd.read_csv("../original_rounds/rounds_1_5.csv", low_memory = False)
val_data_6 = pd.read_csv("../original_rounds/round_6.csv", low_memory = False)

train_data_to_6 = pd.read_csv("../original_rounds/rounds_1_6.csv", low_memory = False)
val_data_7 = pd.read_csv("../original_rounds/round_7.csv", low_memory = False)

train_data_to_7 = pd.read_csv("../original_rounds/rounds_1_7.csv", low_memory = False)
val_data_8 = pd.read_csv("../original_rounds/round_8.csv", low_memory = False)

train_data = pd.read_csv("train.csv",low_memory = False)
test_data = pd.read_csv("test.csv", low_memory=False)

# Data Processing 

In [5]:
def seperate_num_cat(train_data,test_data ):
    data_train = train_data[train_data['employed_status'].notna()].copy()
    output_train = data_train['employed_status'].astype(int)
    data_test = test_data[test_data['employed_status'].notna()].copy()
    output_test = data_test['employed_status'].astype(int)

    numerical = ['age','work_readiness_score','years_unemployed','household_size','total_historical_rounds','tenure_lag','survey_round']
    category= ['gender','province','district','municipality','education_level','has_internet_access','sa_citizen','race','school_quintile','status_broad_lag','employed_lag','status_lag']

    num_columns = [c for c in numerical if c in train_data.columns]
    cat_columns = [c for c in category if c in train_data.columns]

    total_col = num_columns + cat_columns
    dummy_train = data_train[total_col].copy()
    dummy_test = data_test[total_col].copy()

    for x in (dummy_train, dummy_test):
        for c in cat_columns:
            x[c] = x[c].astype(str)

    return output_train, output_test, dummy_train, dummy_test, num_columns, cat_columns

def cat_pipeline():
    return Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='Missing')),('oh',OneHotEncoder(handle_unknown='ignore'))])

def num_pipeline():
    return Pipeline([('impute', SimpleImputer(strategy='median')),('scale',StandardScaler())])

def build(num_columns, cat_columns):
    preprocessor = ColumnTransformer([('cat',cat_pipeline(),cat_columns),('num', num_pipeline(),num_columns),])
    return preprocessor


# testing (1-5) full model

In [6]:
train_output,test_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat(train_data_to_5, val_data_6)
pre = build(num_columns, cat_columns)

## baseline

logistic_model =Pipeline([('pre', pre),('clf', LogisticRegression(max_iter=1000))])
logistic_model.fit(dummy_train, train_output)
log_model_val_prob = logistic_model.predict_proba(dummy_test)[:,1]
log_model_auc = roc_auc_score(test_output,log_model_val_prob)
print(f'log regression val AUC: {log_model_auc:.4f}')

log regression val AUC: 0.6029


In [7]:
grad_boost = Pipeline([('pre', pre),('clf', GradientBoostingClassifier(random_state=42))])
grad_boost.fit(dummy_train, train_output)
grad_val_prob = grad_boost.predict_proba(dummy_test)[:,1]
gra_auc = roc_auc_score(test_output, grad_val_prob)
print(f'grad boost val AUC: {gra_auc:.4f}')

grad boost val AUC: 0.6137


In [8]:
permutation = permutation_importance(logistic_model,dummy_test,test_output,scoring='roc_auc',n_repeats=10,random_state=42)
perm_df = pd.DataFrame({'feature' : dummy_test.columns, 'imp_mean': permutation.importances_mean, 'imp_std': permutation.importances_std})
print(perm_df)

                    feature  imp_mean   imp_std
0                       age  0.012773  0.003267
1      work_readiness_score  0.010728  0.001922
2   total_historical_rounds  0.008835  0.001955
3                tenure_lag  0.000861  0.000411
4                    gender  0.040539  0.005055
5                  province  0.007412  0.002886
6                  district  0.009234  0.002700
7              municipality  0.009976  0.002328
8           education_level -0.001051  0.003390
9                sa_citizen -0.000077  0.000294
10                     race -0.000077  0.000767
11          school_quintile -0.000257  0.001935
12         status_broad_lag  0.027562  0.003849
13             employed_lag  0.016488  0.002515
14               status_lag  0.007890  0.002171


# with dropping columns (negative)

In [9]:
drop = ['education_level','race','school_quintile']
train_data_to_5_reduced_neg = train_data_to_5.drop(columns=drop)
val_data_6_reduced_neg = val_data_6.drop(columns=drop)

train_output,test_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat(train_data_to_5_reduced_neg, val_data_6_reduced_neg)
pre = build(num_columns, cat_columns)

In [10]:
logistic_model =Pipeline([('pre', pre),('clf', LogisticRegression(max_iter=1000))])
logistic_model.fit(dummy_train, train_output)
log_model_val_prob = logistic_model.predict_proba(dummy_test)[:,1]
log_model_auc = roc_auc_score(test_output,log_model_val_prob)
print(f'log regression val AUC: {log_model_auc:.4f}')

log regression val AUC: 0.6092


In [11]:
grad_boost = Pipeline([('pre', pre),('clf', GradientBoostingClassifier(random_state=42))])
grad_boost.fit(dummy_train, train_output)
grad_val_prob = grad_boost.predict_proba(dummy_test)[:,1]
gra_auc = roc_auc_score(test_output, grad_val_prob)
print(f'grad boost val AUC: {gra_auc:.4f}')

grad boost val AUC: 0.6166


# negative and 0s

In [12]:
drop = ['province','education_level','race','school_quintile','tenure_lag','province','sa_citizen','total_historical_rounds']
train_data_to_5_reduced_all = train_data_to_5.drop(columns=drop)
val_data_6_reduced_all = val_data_6.drop(columns=drop)

train_output,test_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat(train_data_to_5_reduced_all, val_data_6_reduced_all)
pre = build(num_columns, cat_columns)

logistic_model =Pipeline([('pre', pre),('clf', LogisticRegression(max_iter=1000))])
logistic_model.fit(dummy_train, train_output)
log_model_val_prob = logistic_model.predict_proba(dummy_test)[:,1]
log_model_auc = roc_auc_score(test_output,log_model_val_prob)
print(f'log regression val AUC: {log_model_auc:.4f}')

grad_boost = Pipeline([('pre', pre),('clf', GradientBoostingClassifier(random_state=42))])
grad_boost.fit(dummy_train, train_output)
grad_val_prob = grad_boost.predict_proba(dummy_test)[:,1]
gra_auc = roc_auc_score(test_output, grad_val_prob)
print(f'grad boost val AUC: {gra_auc:.4f}')

log regression val AUC: 0.6118
grad boost val AUC: 0.6213


# rounds up to 6

In [13]:
drop = ['province','education_level','race','school_quintile','tenure_lag','province','sa_citizen','total_historical_rounds']
train_data_to_6_reduced= train_data_to_6.drop(columns=drop)
val_data_7_reduced = val_data_7.drop(columns=drop)

train_output,test_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat(train_data_to_6_reduced, val_data_7_reduced)
pre = build(num_columns, cat_columns)

logistic_model =Pipeline([('pre', pre),('clf', LogisticRegression(max_iter=1000))])
logistic_model.fit(dummy_train, train_output)
log_model_val_prob = logistic_model.predict_proba(dummy_test)[:,1]
log_model_auc = roc_auc_score(test_output,log_model_val_prob)
print(f'log regression val AUC: {log_model_auc:.4f}')

grad_boost = Pipeline([('pre', pre),('clf', GradientBoostingClassifier(random_state=42))])
grad_boost.fit(dummy_train, train_output)
grad_val_prob = grad_boost.predict_proba(dummy_test)[:,1]
gra_auc = roc_auc_score(test_output, grad_val_prob)
print(f'grad boost val AUC: {gra_auc:.4f}')

log regression val AUC: 0.6371
grad boost val AUC: 0.6383


# up to 7

In [14]:
train_output,test_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat(train_data_to_7, val_data_8)
pre = build(num_columns, cat_columns)

logistic_model =Pipeline([('pre', pre),('clf', LogisticRegression(max_iter=1000))])
logistic_model.fit(dummy_train, train_output)
log_model_val_prob = logistic_model.predict_proba(dummy_test)[:,1]
log_model_auc = roc_auc_score(test_output,log_model_val_prob)
print(f'log regression val AUC: {log_model_auc:.4f}')

grad_boost = Pipeline([('pre', pre),('clf', GradientBoostingClassifier(random_state=42))])
grad_boost.fit(dummy_train, train_output)
grad_val_prob = grad_boost.predict_proba(dummy_test)[:,1]
gra_auc = roc_auc_score(test_output, grad_val_prob)
print(f'grad boost val AUC: {gra_auc:.4f}')

log regression val AUC: 0.6502
grad boost val AUC: 0.6621


In [15]:
permutation = permutation_importance(logistic_model,dummy_test,test_output,scoring='roc_auc',n_repeats=10,random_state=42)
perm_df = pd.DataFrame({'feature' : dummy_test.columns, 'imp_mean': permutation.importances_mean, 'imp_std': permutation.importances_std})
print(perm_df)

                    feature  imp_mean   imp_std
0                       age  0.002006  0.001777
1      work_readiness_score  0.010289  0.004154
2   total_historical_rounds -0.000668  0.000212
3                tenure_lag  0.001421  0.000465
4                    gender  0.090274  0.011540
5                  province  0.018911  0.005129
6                  district  0.008253  0.003186
7              municipality  0.000271  0.004951
8           education_level  0.005611  0.002979
9                sa_citizen -0.000140  0.000176
10                     race  0.004609  0.001666
11          school_quintile  0.000772  0.001725
12         status_broad_lag  0.014120  0.005418
13             employed_lag  0.030597  0.006869
14               status_lag  0.023695  0.004904


In [ ]:
drop = ['sa_citizen','total_historical_rounds','school_quintile','school_quintile']
train_data_to_7_reduced = train_data_to_7.drop(columns=drop)
val_data_8_reduced = val_data_8.drop(columns=drop)

train_output,test_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat(train_data_to_7_reduced, val_data_8_reduced)
pre = build(num_columns, cat_columns)

logistic_model =Pipeline([('pre', pre),('clf', LogisticRegression(max_iter=1000))])
logistic_model.fit(dummy_train, train_output)
log_model_val_prob = logistic_model.predict_proba(dummy_test)[:,1]
log_model_auc = roc_auc_score(test_output,log_model_val_prob)
print(f'log regression val AUC: {log_model_auc:.4f}')

grad_boost = Pipeline([('pre', pre),('clf', GradientBoostingClassifier(random_state=42))])
grad_boost.fit(dummy_train, train_output)
grad_val_prob = grad_boost.predict_proba(dummy_test)[:,1]
gra_auc = roc_auc_score(test_output, grad_val_prob)
print(f'grad boost val AUC: {gra_auc:.5f}')

log regression val AUC: 0.6502
grad boost val AUC: 0.66367


In [ ]:
def seperate_num_cat_test(train_data,test_data):
    data_train = train_data[train_data['employed_status'].notna()].copy()
    output_train = data_train['employed_status'].astype(int)
    data_test = test_data.copy()
    

    numerical = ['age','work_readiness_score','years_unemployed','household_size','total_historical_rounds','tenure_lag','survey_round']
    category= ['gender','province','district','municipality','education_level','has_internet_access','sa_citizen','race','school_quintile','status_broad_lag','employed_lag','status_lag']

    num_columns = [c for c in numerical if c in train_data.columns]
    cat_columns = [c for c in category if c in train_data.columns]

    total_col = num_columns + cat_columns
    dummy_train = data_train[total_col].copy()
    dummy_test = data_test[total_col].copy()
    for x in (dummy_train, dummy_test):
        for c in cat_columns:
            x[c] = x[c].where(x[c].notna(), 'Missing').astype(str)

    return output_train, dummy_train, dummy_test, num_columns, cat_columns

In [ ]:
drop = ['sa_citizen','total_historical_rounds','school_quintile']
train_reduced = train_data.drop(columns=drop)
test_reduced = test_data.drop(columns=drop)

train_output, dummy_train, dummy_test, num_columns, cat_columns = seperate_num_cat_test(train_reduced, test_reduced)
pre = build(num_columns, cat_columns)

grad_boost = Pipeline([('pre', pre),('clf', GradientBoostingClassifier(random_state=42))])
grad_boost.fit(dummy_train, train_output)
grad_val_prob = grad_boost.predict_proba(dummy_test)[:,1]


In [ ]:
logistic_model =Pipeline([('pre', pre),('clf', LogisticRegression(max_iter=1000))])
logistic_model.fit(dummy_train, train_output)
log_model_val_prob = logistic_model.predict_proba(dummy_test)[:,1]

In [ ]:
id = val_data_8['anonymised_id'].copy()
preds = pd.DataFrame({'anonymised_id':id, 'employed_status': grad_val_prob})
preds2 = pd.DataFrame({'anonymised_id':id, 'employed_status': log_model_val_prob})
preds.to_csv('../../../../submission_upgraded_gb.csv', index=False)
preds2.to_csv('../../../../submission_upgraded_log.csv', index=False)